In [ ]:

#  1. Install Required Packages
!pip install mujoco imageio pillow stable-baselines3[extra] shimmy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
#  2. Moun Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving 22.tar.gz to 22.tar.gz


In [ ]:
!tar -xzf /content/22.tar.gz

In [ ]:

# ✅ 4. Configure Headless EGL Rendering
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
#  5. Import Libraries
import mujoco
from mujoco import MjModel, MjData
import numpy as np
import imageio
from PIL import Image
from stable_baselines3 import TD3
from stable_baselines3.common.callbacks import BaseCallback
import gymnasium as gym
from gymnasium import spaces, Env


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
#  6. Load Your Robot XML Model
model = MjModel.from_xml_path("/content/allegro_hand_pick_package/assets/g1_with_hands_torso_only_static.xml")
data = MjData(model)

In [ ]:
!ls /content/allegro_hand_pick_package/assets/

 allegro_hand_pick.xml
 assets
 cube1.xml
 cube_copy.xml
 cube.xml
'g1_dual_arm _copy.xml'
 g1_dual_arm_fixed_hands.xml
 g1_dual_arm_friction_ready.xml
 g1_dual_arm_patched.xml
 g1_dual_arm_powered_hands.xml
 g1_dual_arm_RIGHT_ONLY_PATCHED.xml
 g1_dual_arm_RIGHT_ONLY.xml
 g1_dual_arm_simplified_FIXED_PATCHED.xml
 g1_dual_arm_simplified_fixed.xml
 g1_dual_arm_simplified_PATCHED_GRIP.xml
 g1_dual_arm_simplified.xml
 g1_dual_arm_with_cube.xml
 g1_dual_arm.xml
 g1_with_hands_torso_only_static.xml
 g1_with_hands_torso_only.xml
 g1_with_hands.xml
 left_hand-old.xml
 left_hand.xml
 meshes
 right_hand.xml
 scene_left.xml


In [ ]:
#  Optional: Lighting / Visual Tweaks
model.vis.quality.shadowsize = 4096
model.vis.global_.offwidth = 640
model.vis.global_.offheight = 480

In [ ]:
for i in range(model.nbody):
    print(i, model.body(i).name)

0 world
1 cube
2 pelvis
3 waist_yaw_link
4 waist_roll_link
5 torso_link
6 left_shoulder_pitch_link
7 left_shoulder_roll_link
8 left_shoulder_yaw_link
9 left_elbow_link
10 left_wrist_roll_link
11 left_wrist_pitch_link
12 left_wrist_yaw_link
13 left_hand_thumb_0_link
14 left_hand_thumb_1_link
15 left_hand_thumb_2_link
16 left_hand_middle_0_link
17 left_hand_middle_1_link
18 left_hand_index_0_link
19 left_hand_index_1_link
20 right_shoulder_pitch_link
21 right_shoulder_roll_link
22 right_shoulder_yaw_link
23 right_elbow_link
24 right_wrist_roll_link
25 right_wrist_pitch_link
26 right_wrist_yaw_link
27 right_hand_thumb_0_link
28 right_hand_thumb_1_link
29 right_hand_thumb_2_link
30 right_hand_middle_0_link
31 right_hand_middle_1_link
32 right_hand_index_0_link
33 right_hand_index_1_link


In [ ]:
class CustomRobotEnv(Env):
    def __init__(self, render_mode=None, eval_mode=False):
        super().__init__()
        self.render_mode = render_mode
        self.eval_mode = eval_mode
        self.model = model
        self.data = data

        # Actuators: right side only
        right_actuators = []
        for i in range(self.model.nu):
            name = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
            if name is not None and name.startswith("right_"):
                right_actuators.append(i)
        self.right_actuator_ids = np.array(right_actuators, dtype=np.int32)

        self.action_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(len(self.right_actuator_ids),),
            dtype=np.float32
        )

        self.renderer = mujoco.Renderer(self.model, width=640, height=480)

        obs_dim = self.model.nq + self.model.nv + 9
        self.observation_space = spaces.Box(
            low=-1e10, high=1e10,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.current_step = 0
        self.max_steps = 500
        self.success_counter = 0
        self.freeze_timer = 0

    def reset(self, seed=None, options=None):
      mujoco.mj_resetData(self.model, self.data)
      self.current_step = 0
      super().reset(seed=seed)

      mujoco.mj_forward(self.model, self.data)

      # Set fixed cube spawn (example pos)
      cube_joint_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, "cube:joint")
      cube_qpos_addr = self.model.jnt_qposadr[cube_joint_id]

      fixed_cube_pos = np.array([0.18, 0.0, 0.04])  # X, Y, Z — adjust as needed
      fixed_cube_quat = np.array([1, 0, 0, 0])      # neutral orientation

      self.data.qpos[cube_qpos_addr:cube_qpos_addr + 3] = fixed_cube_pos
      self.data.qpos[cube_qpos_addr + 3:cube_qpos_addr + 7] = fixed_cube_quat

      return self._get_obs(), {}



    def step(self, action):
      # Split action
      arm_action = action[:7]
      finger_action = action[7:]

      # Get positions
      cube_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")
      cube_pos = self.data.xpos[cube_id]
      palm_pos = self.data.body("right_hand_index_1_link").xpos
      thumb_pos = self.data.body("right_hand_thumb_2_link").xpos
      index_pos = self.data.body("right_hand_index_1_link").xpos
      middle_pos = self.data.body("right_hand_middle_1_link").xpos

      # Distances
      dist = np.linalg.norm(palm_pos - cube_pos)
      thumb_dist = np.linalg.norm(thumb_pos - cube_pos)
      index_dist = np.linalg.norm(index_pos - cube_pos)
      middle_dist = np.linalg.norm(middle_pos - cube_pos)

      # Contact detection
      thumb_contact = self._is_touching("cube_geom", "right_hand_thumb_2_geom")
      index_contact = self._is_touching("cube_geom", "right_hand_index_1_geom")
      middle_contact = self._is_touching("cube_geom", "right_hand_middle_1_geom")
      num_contacts = sum([thumb_contact, index_contact, middle_contact])

      # Scale actions based on distance
      ARM_SCALE = 0.4 if dist > 0.08 else 0.2
      FINGER_SCALE = 0.7

      # Reset controls
      self.data.ctrl[:] = 0.0

      # Apply scaled actions
      self.data.ctrl[self.right_actuator_ids[:7]] = arm_action * ARM_SCALE
      self.data.ctrl[self.right_actuator_ids[7:]] = finger_action * FINGER_SCALE

      # Grasp assist: encourage closure when 2 fingers are near
      if dist < 0.06 and num_contacts >= 2:
          assist_strength = 0.5
          self.data.ctrl[self.right_actuator_ids[7:]] += assist_strength
          self.data.ctrl[self.right_actuator_ids[7:]] = np.clip(
              self.data.ctrl[self.right_actuator_ids[7:]], -1.0, 1.0
         )
          print("🤝 Grasp assist triggered (≥2 fingers touching)")

      # Step simulation
      mujoco.mj_step(self.model, self.data)
      obs = self._get_obs()
      reward = self._compute_reward()
      self.current_step += 1

      # Termination
      done = (
          dist > 0.5
          or cube_pos[2] < 0.01
          or cube_pos[2] > 1.0
          or self.current_step >= self.max_steps
          )

      return obs, reward, done, False, {}





    def _compute_reward(self):
      cube_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")
      cube_pos = self.data.xpos[cube_id]
      palm_pos = self.data.body("right_hand_index_1_link").xpos

      dist = np.linalg.norm(palm_pos - cube_pos)
      cube_vel = np.linalg.norm(self.data.cvel[cube_id])

      # Count how many fingers are touching the cube
      fingers = [
        "right_hand_thumb_2_link",
        "right_hand_index_1_link",
        "right_hand_middle_1_link"
      ]
      touch_count = sum(self._is_touching(f, "cube") for f in fingers)

      # Grasp quality heuristic
      if touch_count == 0:
          grasp_quality = -1.0
      elif touch_count == 1:
          grasp_quality = 0.1
      elif touch_count == 2:
          grasp_quality = 0.4
      else:  # 3+
          grasp_quality = 0.9 if cube_vel < 0.05 else 0.5

      # Reward components
      reward = 0
      reward += 5.0 / (1.0 + 20 * dist)
      reward += 2.0 if dist < 0.06 else 0
      reward += 10.0 * grasp_quality
      reward -= 2.0 * min(1.0, cube_vel)
      reward -= 0.005  # time penalty

      # Debug
      print(f"[step {self.current_step}] dist: {dist:.3f}, vel: {cube_vel:.3f}, touches: {touch_count}, grasp_quality: {grasp_quality:.2f}, reward: {reward:.2f}")

      return reward





    def _get_obs(self):
          cube_pos = self.data.body("cube").xpos.copy()
          palm_pos = self.data.body("right_hand_index_1_link").xpos.copy()
          relative_pos = cube_pos - palm_pos
          base_state = np.concatenate([self.data.qpos, self.data.qvel])
          obs = np.concatenate([base_state, cube_pos, palm_pos, relative_pos])
          return obs
    def _is_touching(self, geom1, geom2):
      for i in range(self.data.ncon):
        contact = self.data.contact[i]
        name1 = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_GEOM, contact.geom1)
        name2 = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_GEOM, contact.geom2)
        if (geom1 in (name1, name2)) and (geom2 in (name1, name2)):
            return True
      return False
    def _fingers_touching_cube(self):
      fingers = ["right_hand_thumb_2_link", "right_hand_index_1_link", "right_hand_middle_1_link"]
      touched = 0
      for f in fingers:
          if self._is_touching(f, "cube"):
              touched += 1
      return touched




In [ ]:
env = CustomRobotEnv()
print("Actuator count:", env.model.nu)
print("Right actuator IDs:", env.right_actuator_ids)


Actuator count: 14
Right actuator IDs: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]


In [ ]:
import mujoco
from mujoco import MjModel

# Load your model
model = MjModel.from_xml_path("/content/allegro_hand_pick_package/assets/g1_with_hands_torso_only_static.xml")

print("=== Actuator names in the model ===")
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"{i}: {name}")


=== Actuator names in the model ===
0: right_shoulder_pitch_joint
1: right_shoulder_roll_joint
2: right_shoulder_yaw_joint
3: right_elbow_joint
4: right_wrist_roll_joint
5: right_wrist_pitch_joint
6: right_wrist_yaw_joint
7: right_hand_thumb_0_joint
8: right_hand_thumb_1_joint
9: right_hand_thumb_2_joint
10: right_hand_index_0_joint
11: right_hand_index_1_joint
12: right_hand_middle_0_joint
13: right_hand_middle_1_joint


In [ ]:
#  8. Define a Simple Training Log Callback
class PrintCallback(BaseCallback):
    def __init__(self, verbose=1):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        if self.n_calls % 100 == 0:
            print(f"Step {self.n_calls}, reward: {self.locals['rewards'][-1]:.3f}")
        return True

In [ ]:
#  1. Import the noise module
from stable_baselines3.common.noise import NormalActionNoise

#  2. Create the environment first
env = CustomRobotEnv()

#  3. Now define the noise after env is available
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.3 * np.ones(n_actions))

#  4. Create the SAC model with noise
model_sac = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
import os
import imageio
from stable_baselines3.common.callbacks import BaseCallback
from datetime import datetime
from PIL import Image

class EvalVideoCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=50000, video_length=300, video_folder="videos/", prefix="grasp_eval", verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.video_length = video_length
        self.video_folder = video_folder
        self.prefix = prefix
        os.makedirs(video_folder, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            obs, _ = self.eval_env.reset()
            frames = []

            for _ in range(self.video_length):
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, done, _, _ = self.eval_env.step(action)

                self.eval_env.renderer.update_scene(self.eval_env.data)
                frame = self.eval_env.renderer.render()
                frames.append(Image.fromarray(frame.astype(np.uint8)))

                if done:
                    break

            # Save video
            timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
            video_path = os.path.join(
                self.video_folder, f"{self.prefix}_{self.n_calls}_steps_{timestamp}.mp4"
            )
            imageio.mimsave(video_path, frames, fps=30)
            print(f"🎥 Saved evaluation video: {video_path}")

        return True


In [ ]:
env = CustomRobotEnv()
eval_env = CustomRobotEnv()

callback = EvalVideoCallback(
    eval_env=eval_env,
    eval_freq=50000,
    video_length=300,
    video_folder="/content/videos/",
    prefix="grasp_eval"
)

model_sac = TD3(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    batch_size=256,
    buffer_size=1_000_000,
    gamma=0.98,
    tau=0.02
)

model_sac.learn(total_timesteps=50_000, callback=callback)


# Save model
model_sac.save("/content/drive/MyDrive/sac_custom_robot")


Streaming output truncated to the last 5000 lines.
[step 53] dist: 0.241, vel: 0.019, touches: 0, grasp_quality: -1.00, reward: -9.18
[step 54] dist: 0.241, vel: 0.017, touches: 0, grasp_quality: -1.00, reward: -9.18
[step 55] dist: 0.240, vel: 0.016, touches: 0, grasp_quality: -1.00, reward: -9.18
[step 56] dist: 0.240, vel: 0.014, touches: 0, grasp_quality: -1.00, reward: -9.17
[step 57] dist: 0.240, vel: 0.013, touches: 0, grasp_quality: -1.00, reward: -9.17
[step 58] dist: 0.240, vel: 0.012, touches: 0, grasp_quality: -1.00, reward: -9.17
[step 59] dist: 0.240, vel: 0.011, touches: 0, grasp_quality: -1.00, reward: -9.16
[step 60] dist: 0.240, vel: 0.010, touches: 0, grasp_quality: -1.00, reward: -9.16
[step 61] dist: 0.240, vel: 0.009, touches: 0, grasp_quality: -1.00, reward: -9.16
[step 62] dist: 0.240, vel: 0.008, touches: 0, grasp_quality: -1.00, reward: -9.16
[step 63] dist: 0.240, vel: 0.007, touches: 0, grasp_quality: -1.00, reward: -9.16
[step 64] dist: 0.240, vel: 0.006, t

In [ ]:
# 10. Evaluate and Record Video (longer)
frames = []
obs, _ = env.reset()
for t in range(1000):  # increase number of steps/frames
    action, _ = model_sac.predict(obs, deterministic=True)
    obs, _, _, _, _ = env.step(action)
    env.renderer.update_scene(env.data)
    frame = env.renderer.render()
    frames.append(Image.fromarray(frame.astype(np.uint8)))

# Save at 30 fps
imageio.mimsave("custom_robot_eval.mp4", frames, fps=30)

[step 0] dist: 0.254, vel: 0.000, touches: 0, grasp_quality: -1.00, reward: -9.18
[step 1] dist: 0.254, vel: 0.020, touches: 0, grasp_quality: -1.00, reward: -9.22
[step 2] dist: 0.254, vel: 0.039, touches: 0, grasp_quality: -1.00, reward: -9.26
[step 3] dist: 0.253, vel: 0.059, touches: 0, grasp_quality: -1.00, reward: -9.30
[step 4] dist: 0.253, vel: 0.078, touches: 0, grasp_quality: -1.00, reward: -9.34
[step 5] dist: 0.253, vel: 0.098, touches: 0, grasp_quality: -1.00, reward: -9.38
[step 6] dist: 0.253, vel: 0.118, touches: 0, grasp_quality: -1.00, reward: -9.41
[step 7] dist: 0.253, vel: 0.137, touches: 0, grasp_quality: -1.00, reward: -9.45
[step 8] dist: 0.253, vel: 0.157, touches: 0, grasp_quality: -1.00, reward: -9.49
[step 9] dist: 0.252, vel: 0.177, touches: 0, grasp_quality: -1.00, reward: -9.53
[step 10] dist: 0.252, vel: 0.196, touches: 0, grasp_quality: -1.00, reward: -9.57
[step 11] dist: 0.252, vel: 0.216, touches: 0, grasp_quality: -1.00, reward: -9.61
[step 12] dist

In [21]:
# Download the video
video_path = "custom_robot_eval.mp4"
files.download(video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>